# Maximum Hosting Capacity: Input Data Pipeline

**Project:** NEXT-GW Subproject 2.2, uncertainty-aware hosting capacity
**Based on:** meeting with Yifan, October 7, 2026
**Scope:** we start with the **Eastern Interconnection**

### How to use this notebook

The notebook is split into sections. Text sections explain the project, and code sections (the grey boxes) do the work. Run code boxes **from top to bottom**, one at a time, by clicking a box and pressing **Shift + Enter**. Each box prints what it did, so you can check the result before moving on.

Boxes marked **(demo)** run on small made-up numbers, so you can see how a step works without downloading anything. Boxes marked **(live)** download real data and are switched off until we turn them on.

## 1. The big picture

The question this project answers is: **how much new load $C_i$ can we add at bus $i$ without breaking the grid, in every hour of the year?** That number is the bus's *maximum hosting capacity*. A new load could be a data center, an EV charging hub, or a factory.

The answer has to hold in all 8,760 hours of a year (365 × 24). Demand and renewable output change every hour, so the grid has more room in some hours than in others. Because the new load runs all year, it can only be as big as the **tightest** hour allows.

Solar and wind output is uncertain. A cloudy, windless hour leaves less free power to serve the new load, so the model needs realistic hourly renewable data to find those tight hours. Later, the group will run many demand and renewable *scenarios* to see how hosting capacity changes under that uncertainty.

**My role** is to build the input data, as CSV files, that the model reads.

## 2. The model

The grid has $N$ buses (points where power enters or leaves) and $M$ lines connecting them. Bold letters are vectors with one entry per bus, for one hour $t$.

### Objective

We want to make the new load at bus $i$ as large as possible. The model is free to choose how much each generator and renewable plant produces each hour to make room for it.

$$\max_{\,\mathbf{g}_t,\ \mathbf{r}_t,\ C_i} \; C_i$$

### Constraint 1: generator limits

Each generator produces between zero and its maximum, in every hour. For solar and wind, the maximum changes every hour with the weather. That hourly maximum, $\mathbf{r}_t^{\max}$, is the renewable data this notebook collects.

$$0 \le \mathbf{g}_t \le \mathbf{g}_t^{\max}, \qquad 0 \le \mathbf{r}_t \le \mathbf{r}_t^{\max}, \qquad \forall\, t \in \mathcal{T} = \{1, \dots, 8760\}$$

### Constraint 2: power balance

The total power produced must equal the total power used, every hour. The $\mathbf{1}^\top$ simply adds up all buses. The vector $\mathbf{c}$ is all zeros except for $C_i$ at bus $i$, because the new load sits only there.

$$\mathbf{1}^\top \left( \mathbf{g}_t + \mathbf{r}_t - \mathbf{d}_t - \mathbf{c} \right) = 0$$

### Constraint 3: line flow limits

The part in parentheses is each bus's net injection (power in minus power out). The matrix $H$ turns those injections into the power flowing on each line, and every line's flow must stay within its rating $\bar{\mathbf{f}}$ in both directions. $H$ is built from the line list: which two buses each line connects, and its impedance. In power systems it is called the PTDF (power transfer distribution factor) matrix.

$$-\bar{\mathbf{f}} \;\le\; H \left( \mathbf{g}_t + \mathbf{r}_t - \mathbf{d}_t - \mathbf{c} \right) \;\le\; \bar{\mathbf{f}}$$

### Symbol cheat sheet

| Symbol | Meaning | Size | Where it comes from |
|---|---|---|---|
| $t$, $\mathcal{T}$ | One hour; the set of all 8,760 hours | — | Calendar year |
| $C_i$ | New load added at bus $i$ (the answer) | one number, MW | Solved by the model |
| $\mathbf{c}$ | Vector with $C_i$ at bus $i$, zeros elsewhere | $N$ | Built from $C_i$ |
| $\mathbf{g}_t$ | Conventional generator output each hour | $N$ | Decided by the model |
| $\mathbf{g}_t^{\max}$ | Generator capacity | $N$ | Generator data |
| $\mathbf{r}_t$ | Renewable output each hour | $N$ | Decided by the model |
| $\mathbf{r}_t^{\max}$ | Maximum solar + wind available that hour | $N \times 8760$ | **This notebook** |
| $\mathbf{d}_t$ | Existing demand each hour | $N \times 8760$ | **This notebook** |
| $H$ | Turns bus injections into line flows | $M \times N$ | Built from line data |
| $\bar{\mathbf{f}}$ | Line flow limit (rating) | $M$ | **This notebook** |

## 3. The US grid: a guide for newcomers

The US power grid is organized like nested boxes, from biggest to smallest:

**Interconnection → Grid operator (ISO/RTO) → Balancing authority (BA) → Zone → Bus**

### Interconnections

There are three separate grids in the US. Power flows freely inside each one, but only a small amount passes between them. Buses in different interconnections are not connected, so each interconnection is modeled on its own.

| Interconnection | Covers |
|---|---|
| **Eastern** (our starting point) | Everything east of the Rocky Mountains, except most of Texas |
| **Western** | The Rocky Mountains to the Pacific coast |
| **ERCOT** | Most of Texas, on its own |

### Grid operators (ISOs and RTOs)

An ISO (Independent System Operator) or RTO (Regional Transmission Organization) is an organization that runs the grid and the power market across a large area. For our purposes the two names mean the same thing. There are seven in the US, and five of them are in the Eastern Interconnection.

| Operator | Covers roughly | Interconnection |
|---|---|---|
| **ISO-NE** | New England | Eastern |
| **NYISO** | New York State | Eastern |
| **PJM** | Mid-Atlantic to Chicago (13 states and Washington DC) | Eastern |
| **MISO** | The Midwest down to Louisiana, including Indiana | Eastern |
| **SPP** | The Great Plains (Kansas, Oklahoma, the Dakotas, and others) | Mostly Eastern; confirm its exact footprint before use |
| **ERCOT** | Texas | ERCOT |
| **CAISO** | Most of California | Western |

### Balancing authorities, zones and buses

Every area of the grid has a **balancing authority (BA)**, whose job is to keep supply and demand matched at every moment. Each ISO is a BA. However, some areas have **no ISO at all**. In the Eastern Interconnection this is mainly the Southeast, where utilities such as Southern Company, Duke Energy, TVA and the Florida utilities act as their own BAs.

Inside an ISO, the area is often divided into smaller **zones** (for example, NYISO has 11 load zones and ISO-NE has 8). The smallest unit is the **bus**, the individual point in our model.

### What this means for our data

EIA-930, from the US Energy Information Administration, covers **every BA in the country**, but only as one total per BA. It is the backbone that fills in areas with no ISO. The ISOs publish finer, zone-level data, but only inside their own area. So we use ISO zone data where it exists, fall back to EIA-930 elsewhere, and record which resolution each number came from.

### Renewable sources found so far (below the whole-ISO level)

| Operator | What is available | Status |
|---|---|---|
| ISO-NE | Solar for 8 zones | Found |
| NYISO | Solar for 11 zones; wind only for all of NYISO | Found |
| MISO | Solar and wind for North, Central and South | Found |
| PJM | Solar and wind by area | Reachable without an account using the public Data Miner 2 key method from the instructor's script; not yet tested on these feeds |

## 4. The data we deliver

### Two kinds of files

Our data comes in areas of different sizes: NYISO's 11 zones, MISO's 3 regions, PJM's areas, and whole BAs in the Southeast. To hold all of them in one clean format, we use two kinds of files.

The first is an **areas list** (`areas.csv`) with one row per area, written once. It records which interconnection, operator, BA and zone each area belongs to, how fine its resolution is, and where its data came from. This file is also what a US map will use later.

The second is **one data file per data type** (`demand.csv`, `solar.csv`, `wind.csv`). Every row is one hour in one area, with just three columns: `time_utc`, `area_id` and `value_mw`. To learn more about a row, you look up its `area_id` in `areas.csv`. Keeping the details in one place means they can never disagree with each other.

### Folder layout

Each source gets its own small fetcher. A fetcher saves the raw download untouched, converts the time to UTC, renames the columns to the standard format, and adds its rows to the shared files. If one source breaks, only its fetcher needs fixing.

```
data/
  raw/    pjm/  nyiso/  miso/  isone/  eia930/   <- downloads, never edited
  clean/  areas.csv  demand.csv  solar.csv  wind.csv
  model/  buses.csv  lines.csv  bus_demand.csv  bus_renewables.csv
```

### What the model finally reads

At the end, each bus is assigned to the area it sits in, and each area's hourly total is split among its buses. That produces the bus-level files in `data/model/`:

| File | Columns | Rows |
|---|---|---|
| `buses.csv` | bus_id, name, latitude, longitude, area_id, voltage_kv | one per bus |
| `lines.csv` | line_id, from_bus, to_bus, reactance_pu, flow_limit_mw | one per line |
| `bus_demand.csv` | hour, bus_id, demand_mw | 8,760 × number of buses |
| `bus_renewables.csv` | hour, bus_id, solar_max_mw, wind_max_mw | 8,760 × number of buses |

## 5. Time zones

Every timestamp is stored in **UTC**. UTC never changes for daylight saving, so a year is always exactly 8,760 clean hours. Local time, by contrast, has one 23-hour day in spring and one 25-hour day in fall.

An earlier mistake put MISO's values one hour off. It happened because hours were added by hand and MISO's time label meant something different from what we assumed. To prevent this, we write down exactly what each source's time label means, and we let code do the conversion:

| Source label | What it means | Offset from UTC |
|---|---|---|
| PJM "EPT" | Eastern Prevailing Time, follows daylight saving | UTC−4 in summer, UTC−5 in winter |
| MISO "EST" | Eastern Standard Time, fixed all year | UTC−5 always |

After converting, we check three things automatically: there are exactly 8,760 unique hours, there are no duplicate or missing hours, and solar is zero at night. The night-time solar check is what caught the MISO error last time.

## 6. Setup

This box loads the Python libraries and sets the main settings. **Run it first**, every time you open the notebook. The EIA key is read from your computer's `EIA_API_KEY` setting, or from a `.env` file next to this notebook containing the line `EIA_API_KEY=your_key`. Never paste the key into the notebook itself, and keep `.env` out of git. The year and region are placeholders until we confirm them with Yifan.

In [1]:
import os
import time

import pandas as pd
import requests

YEAR = 2023                    # which full year to build (placeholder, confirm with Yifan)
INTERCONNECTION = "Eastern"    # our starting scope
DATA_DIR = "data"              # where all files are saved
RUN_LIVE = False               # set to True only when we are ready to download real data

EIA_API_KEY = os.environ.get("EIA_API_KEY", "")   # free key from eia.gov/opendata
if not EIA_API_KEY and os.path.exists(".env"):     # or keep it in a .env file next to this notebook
    for line in open(".env"):
        if line.strip().startswith("EIA_API_KEY="):
            EIA_API_KEY = line.split("=", 1)[1].strip().strip("'\"")
EIA_BASE = "https://api.eia.gov/v2"

for folder in ["raw", "clean", "model"]:
    os.makedirs(os.path.join(DATA_DIR, folder), exist_ok=True)

print(f"Setup done. Year {YEAR}, {INTERCONNECTION} Interconnection, live downloads: {RUN_LIVE}")

Setup done. Year 2023, Eastern Interconnection, live downloads: False


## 7. Converting time to UTC

This function converts a column of local times to UTC using the time zone rules built into Python, so daylight saving is handled correctly. The `source_tz` argument says what the source's label means:

- `"America/New_York"` for **prevailing** Eastern time (PJM's EPT), which follows daylight saving.
- `"Etc/GMT+5"` for **fixed** Eastern Standard Time (MISO's EST). The plus sign is a quirk of this naming system; it really means UTC−5.

If the table holds several areas, each area is converted separately (by its `area_id` column), because mixing areas confuses the daylight-saving step.

In [2]:
def to_utc(df, time_col, source_tz, group_col="area_id"):
    # Convert a column of local times (no time zone attached) to UTC.
    if group_col in df.columns:
        # Convert each area on its own: with several areas mixed together, the repeated
        # fall hour appears many times in a row and 'infer' below can't tell them apart.
        parts = [to_utc(part, time_col, source_tz, group_col=None) for _, part in df.groupby(group_col, sort=False)]
        return pd.concat(parts, ignore_index=True)
    local = pd.to_datetime(df[time_col])
    if source_tz == "America/New_York":
        # In fall, one local hour happens twice; 'infer' uses the order of the rows to tell them apart.
        local = local.dt.tz_localize(source_tz, ambiguous="infer", nonexistent="shift_forward")
    else:
        local = local.dt.tz_localize(source_tz)
    out = df.drop(columns=time_col).copy()
    out.insert(0, "time_utc", local.dt.tz_convert("UTC"))
    return out

print("to_utc() is ready.")

to_utc() is ready.


**(demo)** The same local clock time, 12:00 on a summer day, converted from each source. PJM's EPT is in daylight saving time in July, so it becomes 16:00 UTC. MISO's fixed EST becomes 17:00 UTC. Mixing these up is exactly what caused the one-hour MISO error.

In [3]:
example = pd.DataFrame({"local_time": ["2023-07-01 12:00"], "value_mw": [100.0]})

print("PJM  (EPT):", to_utc(example, "local_time", "America/New_York")["time_utc"].iloc[0])
print("MISO (EST):", to_utc(example, "local_time", "Etc/GMT+5")["time_utc"].iloc[0])

PJM  (EPT): 2023-07-01 16:00:00+00:00
MISO (EST): 2023-07-01 17:00:00+00:00


### Checking a full year

This function confirms that a dataset has every hour of the year exactly once, for every area. It reports any missing or duplicate hours so they can be fixed before the data reaches the model.

A second function checks the values themselves: no blank or negative numbers, and for solar, no output at night. Solar at night almost always means the time zone was converted wrong.

In [4]:
def check_full_year(df, year, group_col="area_id"):
    expected = pd.date_range(f"{year}-01-01 00:00", f"{year}-12-31 23:00", freq="h", tz="UTC")
    all_good = True
    for name, part in df.groupby(group_col):
        times = part["time_utc"]
        duplicates = int(times.duplicated().sum())
        missing = len(expected.difference(pd.DatetimeIndex(times)))
        if duplicates or missing:
            all_good = False
            print(f"  {name}: {missing} missing hours, {duplicates} duplicate hours")
    print("Every area has all 8,760 hours exactly once." if all_good else "Fix the problems above.")


def check_values(df, kind):
    # kind is "demand", "solar" or "wind". Looks for blank, negative and (for solar) night-time values.
    all_good = True
    for name, part in df.groupby("area_id"):
        values = part["value_mw"]
        blank, negative = int(values.isna().sum()), int((values < 0).sum())
        problems = []
        if blank:
            problems.append(f"{blank} blank values")
        if negative:
            problems.append(f"{negative} negative values")
        if kind == "solar":
            # 04:00-08:00 UTC is night everywhere in the Eastern Interconnection, all year
            # (11 pm to 4 am Eastern, 10 pm to 3 am Central).
            night = values[part["time_utc"].dt.hour.between(4, 8)]
            sunny_nights = int((night > 0.01 * values.max()).sum())
            if sunny_nights:
                problems.append(f"{sunny_nights} night hours with solar (time zone may be off)")
        if problems:
            all_good = False
            print(f"  {name}: " + ", ".join(problems))
    print(f"All {kind} values look sane." if all_good else "Fix the problems above.")

**(demo)** A made-up area with a full year passes. A second area with one hour removed is caught. Then a made-up solar area with output in the middle of the night is caught too.

In [5]:
hours = pd.date_range(f"{YEAR}-01-01", periods=8760, freq="h", tz="UTC")
good = pd.DataFrame({"time_utc": hours, "area_id": "AREA_OK", "value_mw": 1.0})
bad = good.drop(index=100).assign(area_id="AREA_BROKEN")

check_full_year(pd.concat([good, bad]), YEAR)

solar_shape = (good["time_utc"].dt.hour.isin(range(12, 23))).astype(float) * 50   # sunny 12:00-22:00 UTC
solar_ok = good.assign(area_id="SOLAR_OK", value_mw=solar_shape)
solar_shifted = good.assign(area_id="SOLAR_SHIFTED", value_mw=solar_shape.shift(-10, fill_value=0))  # 10 hours off
check_values(pd.concat([solar_ok, solar_shifted]), "solar")

  AREA_BROKEN: 1 missing hours, 0 duplicate hours
Fix the problems above.
  SOLAR_SHIFTED: 1825 night hours with solar (time zone may be off)
Fix the problems above.


## 8. Fetching data

Each source gets its own function. Every function returns the same three columns, `time_utc`, `area_id` and `value_mw`, so their results can be stacked into one file.

### EIA (US Energy Information Administration)

EIA's website has an API: instead of clicking and downloading files by hand, code asks EIA's server for exactly the data it wants and gets it back as a table. This helper handles that request. EIA sends at most 5,000 rows at a time, so the helper keeps asking for the next 5,000 until it has everything. It needs a free API key, stored on your computer as `EIA_API_KEY`. If a request fails, the error message leaves the key out, so it can't end up saved in the notebook's output.

In [6]:
def eia_get_all(route, params):
    # Download every row from one EIA API route, 5,000 rows at a time.
    # Pages are sorted so they line up; pass extra sort columns in params (e.g. "sort[1][column]").
    params = {"sort[0][column]": "period", "sort[0][direction]": "asc", **params}
    rows, offset = [], 0
    while True:
        request_params = dict(params, api_key=EIA_API_KEY, offset=offset, length=5000)
        try:
            response = requests.get(f"{EIA_BASE}/{route}/data/", params=request_params, timeout=60)
        except requests.RequestException as e:
            # Python's own error message includes the full web address, which contains the API key.
            raise RuntimeError(f"Could not reach EIA: {type(e).__name__}") from None
        if not response.ok:
            message = response.text[:300].replace(EIA_API_KEY, "***") if EIA_API_KEY else response.text[:300]
            raise RuntimeError(f"EIA request failed ({response.status_code}): {message}")
        body = response.json()["response"]
        page = body["data"]
        rows.extend(page)
        if len(page) < 5000:
            break
        offset += 5000
        time.sleep(0.2)   # pause briefly so we don't overload EIA's server
    if len(rows) != int(body["total"]):
        raise RuntimeError(f"EIA said {body['total']} rows but we received {len(rows)}.")
    return pd.DataFrame(rows)

print("eia_get_all() is ready.")

eia_get_all() is ready.


### Demand by subregion (EIA-930)

For some large operators, EIA-930 also reports demand for subregions inside the operator, which is finer than one total. This function requests those subregion values for one operator and one year. **Tested on 2023 MISO data (October 23, 2026):** all 6 subregions had 8,760 hours with no blank or negative values. The times are UTC, but EIA labels each hour by its **end** ("01:00" means 00:00 to 01:00), while we label hours by their start, like PJM does. So the code subtracts one hour. This was found on October 25, 2026 by comparing EIA's MISO solar with MISO's own file (they match exactly once shifted) and by checking that solar peaks at solar noon.

In [7]:
def fetch_subregion_demand(parent_ba, year):
    raw = eia_get_all("electricity/rto/region-sub-ba-data", {
        "frequency": "hourly",
        "data[0]": "value",
        "facets[parent][]": parent_ba,
        "start": f"{year}-01-01T01",   # hour-ending labels: 01:00 is the first hour
        "end": f"{year + 1}-01-01T00",
        "sort[1][column]": "subba",
        "sort[1][direction]": "asc",
    })
    out = pd.DataFrame({
        "time_utc": pd.to_datetime(raw["period"], utc=True) - pd.Timedelta(hours=1),   # EIA labels the END of the hour
        "area_id": parent_ba + "_" + raw["subba"],
        "value_mw": pd.to_numeric(raw["value"]),
    })
    return out

print("fetch_subregion_demand() is ready.")

fetch_subregion_demand() is ready.


**(live)** Downloads one year of MISO subregion demand. It only runs when `RUN_LIVE = True` in the setup box and an EIA key is set.

In [8]:
if RUN_LIVE:
    miso_demand = fetch_subregion_demand("MISO", YEAR)
    miso_demand.to_csv(f"{DATA_DIR}/raw/eia930_miso_subregion_demand_{YEAR}.csv", index=False)
    check_full_year(miso_demand, YEAR)
    check_values(miso_demand, "demand")
else:
    print("Skipped: RUN_LIVE is False.")

Skipped: RUN_LIVE is False.


### Demand, solar and wind for every balancing authority (EIA-930)

EIA-930 also reports, for **every** balancing authority (BA) in the country, its hourly demand and its hourly output by fuel, including solar (`SUN`) and wind (`WND`). That is one total per BA, coarser than the zones some ISOs publish, but it covers the whole Eastern Interconnection, including the Southeast where there is no ISO. Section 3's plan is to use the finer ISO zone data where it exists and these BA totals everywhere else.

`EASTERN_BAS` lists the EIA-930 codes for BAs in the Eastern Interconnection (US only). A BA that reports no solar or wind simply has no rows; the function prints which ones.

In [ ]:
EASTERN_BAS = [
    "AECI", "CPLE", "CPLW", "DUK", "EEI", "FMPP", "FPC", "FPL", "GLHB", "GVL", "HST", "ISNE",
    "JEA", "LGEE", "MISO", "NSB", "NYIS", "PJM", "SC", "SCEG", "SEC", "SEPA", "SOCO", "SPA",
    "SWPP", "TAL", "TEC", "TVA", "YAD",
]


def fetch_eia930_ba(kind, year, bas=EASTERN_BAS):
    # kind is "demand", "solar" or "wind". Returns time_utc, area_id, value_mw, one area per BA.
    if kind == "demand":
        route, facet = "electricity/rto/region-data", {"facets[type][]": "D"}
    else:
        route, facet = "electricity/rto/fuel-type-data", {"facets[fueltype][]": {"solar": "SUN", "wind": "WND"}[kind]}
    raw = eia_get_all(route, {
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": list(bas),
        **facet,
        "start": f"{year}-01-01T01",   # hour-ending labels: 01:00 is the first hour
        "end": f"{year + 1}-01-01T00",
        "sort[1][column]": "respondent",
        "sort[1][direction]": "asc",
    })
    out = pd.DataFrame({
        # EIA labels the END of each hour; we label the start. PJM's own submission to EIA is
        # already labelled by the start (its solar peaks an hour earlier than everyone else's), so it is not shifted.
        "time_utc": pd.to_datetime(raw["period"], utc=True) - pd.to_timedelta((raw["respondent"] != "PJM").astype(int), unit="h"),
        "area_id": "EIA_" + raw["respondent"],
        "value_mw": pd.to_numeric(raw["value"]),
    })
    silent = sorted(set(bas) - set(raw["respondent"]))
    print(f"EIA-930 {kind}: {len(out)} rows, {out['area_id'].nunique()} BAs. No {kind} data from: {', '.join(silent) or 'none'}")
    return out

print("fetch_eia930_ba() is ready (not yet tested on live data).")

### Cleaning the EIA-930 data

EIA-930 is self-reported by each BA, and the 2023 download shows the usual problems: a few missing hours (FPL 24, SCEG 25, SEC 49), impossible spikes (SPP demand of 3.6 million MW for one hour, FPL solar of 41,000 MW at midnight), small negative solar at night, and stuck solar values overnight (ISO-NE reports a flat 28 MW on some nights). This function fixes them in a fixed order and reports what it changed:

1. A value more than twice the area's 99th percentile, or negative demand, is treated as missing.
2. Solar and wind below zero become 0, and solar between 04:00 and 08:00 UTC (night everywhere in the Eastern Interconnection) becomes 0.
3. Every area gets all 8,760 hours. Gaps of up to 6 hours are filled by drawing a straight line between the neighbours. Longer gaps (for example a whole day missing) take the value from the same hour on the nearest day that has one: 1 day before or after, then 2, then a week.
4. An area whose values are all zero is dropped, because zero there means "not reported", not "no output" (for example, NYISO reports no solar to EIA-930).

In [ ]:
def clean_hourly(df, kind, year, max_gap=6):
    # df: time_utc, area_id, value_mw. Returns a cleaned copy with every hour of the year for every area.
    hours = pd.date_range(f"{year}-01-01 00:00", f"{year}-12-31 23:00", freq="h", tz="UTC")
    parts = []
    for area, part in df.groupby("area_id"):
        v = part.drop_duplicates("time_utc").set_index("time_utc")["value_mw"].reindex(hours)
        spikes = v > 2 * v.quantile(0.99)
        if kind == "demand":
            spikes |= v < 0
        v[spikes] = float("nan")
        if kind in ("solar", "wind"):
            v = v.clip(lower=0)
        if kind == "solar":
            v[v.index.hour.isin(range(4, 9))] = 0.0
        blank_before = int(v.isna().sum())
        v = v.interpolate(limit=max_gap, limit_area="inside")
        filled_short = blank_before - int(v.isna().sum())
        for days in [1, -1, 2, -2, 7, -7]:   # longer gaps: same hour on a nearby day
            v = v.fillna(v.shift(24 * days))
        if v.fillna(0).max() == 0:
            print(f"  {area}: all zero, dropped (not reported)")
            continue
        notes = []
        if spikes.sum():
            notes.append(f"{int(spikes.sum())} spikes removed")
        if blank_before:
            notes.append(f"{filled_short} short-gap hours filled by a straight line, "
                         f"{blank_before - filled_short - int(v.isna().sum())} from a nearby day")
        if v.isna().sum():
            notes.append(f"{int(v.isna().sum())} hours still blank")
        if notes:
            print(f"  {area}: " + ", ".join(notes))
        parts.append(pd.DataFrame({"time_utc": hours, "area_id": area, "value_mw": v.values}))
    out = pd.concat(parts, ignore_index=True)
    print(f"Cleaned {kind}: {out['area_id'].nunique()} areas.")
    return out

print("clean_hourly() is ready.")

**(live)** Downloads one year of demand, solar and wind for every Eastern BA, saves the raw download to `data/raw/`, cleans it, saves the cleaned version to `data/clean/`, and runs the checks. It only runs when `RUN_LIVE = True`.

In [ ]:
if RUN_LIVE:
    for kind in ["demand", "solar", "wind"]:
        ba = fetch_eia930_ba(kind, YEAR)
        ba.to_csv(f"{DATA_DIR}/raw/eia930_ba_{kind}_{YEAR}.csv", index=False)
        ba = clean_hourly(ba, kind, YEAR)
        ba.to_csv(f"{DATA_DIR}/clean/eia930_ba_{kind}_{YEAR}.csv", index=False)
        check_full_year(ba, YEAR)
        check_values(ba, kind)
else:
    print("Skipped: RUN_LIVE is False.")

### Solar and wind

This step will produce $\mathbf{r}_t^{\max}$, the maximum solar and wind available at each bus, each hour. Nobody publishes this number per bus; it only exists for bigger areas such as zones. So we have to create the per-bus numbers ourselves, and Section 9 explains the two possible ways to do that (Option A, our plan, and Option B, a question for Yifan).

For Option A, this step downloads hourly solar and wind for the **smallest area** each operator publishes (see the table in Section 3), converts the times to UTC, and saves them in the standard format. Areas with no ISO, such as the Southeast, would use EIA-930's one total per BA as a last resort.

PJM is written below. ISO-NE, NYISO and MISO are not written yet.

#### PJM (Data Miner 2)

PJM publishes hourly solar and wind output by area through its Data Miner 2 API. The API needs a key, but PJM's own Data Miner website uses a public key that anyone can read from its settings file, so no account is needed. This is the method from the instructor's script.

PJM gives each row's start time in both UTC (`datetime_beginning_utc`) and local EPT (`datetime_beginning_ept`). We use the UTC column directly, and we also convert the EPT column with `to_utc()` to make sure the two agree. Non-members may only make a few requests a minute, so the helper waits between pages.

PJM reports six areas, but some are totals of others. On the 2023 data, `RTO` (all of PJM) equals `MIDATL` + `WEST` + `SOUTH`, and `RFC` equals `MIDATL` + `WEST`, to within 0.001 MW. So we keep only **MIDATL, WEST and SOUTH**; keeping the totals would count output twice. `OTHER` is not part of `RTO` (about 5% as much solar, 0.2% as much wind) and is left out until we know what it covers.

Solar plants use a little power at night, which PJM reports as small negative output (down to about −38 MW). Available output can never be below zero, so those values are set to 0.

In [ ]:
PJM_API = "https://api.pjm.com/api/v1"


def pjm_public_key():
    # The public key PJM's Data Miner website uses. It is read fresh each time, never printed or saved.
    response = requests.get("https://dataminer2.pjm.com/config/settings.json", timeout=30)
    response.raise_for_status()
    return response.json()["subscriptionKey"]


def pjm_get_all(feed, params, rows_per_page=50000):
    # Download every row of one Data Miner 2 feed, one page at a time.
    headers = {"Ocp-Apim-Subscription-Key": pjm_public_key()}
    rows, start_row = [], 1
    while True:
        request_params = dict(params, rowCount=rows_per_page, startRow=start_row)
        response = requests.get(f"{PJM_API}/{feed}", params=request_params, headers=headers, timeout=120)
        if not response.ok:
            raise RuntimeError(f"PJM request failed ({response.status_code}): {response.text[:300]}")
        body = response.json()
        rows.extend(body["items"])
        if start_row - 1 + len(body["items"]) >= body["totalRows"] or not body["items"]:
            break
        start_row += rows_per_page
        time.sleep(12)   # non-members may only make a few requests a minute
    if len(rows) != body["totalRows"]:
        raise RuntimeError(f"PJM said {body['totalRows']} rows but we received {len(rows)}.")
    return pd.DataFrame(rows)


def fetch_pjm_renewables(kind, year):
    # kind is "solar" or "wind". Returns time_utc, area_id, value_mw for every PJM area.
    raw = pjm_get_all(f"{kind}_gen", {
        "datetime_beginning_utc": f"1/1/{year} 00:00to12/31/{year} 23:00",
        "fields": f"datetime_beginning_utc,datetime_beginning_ept,area,{kind}_generation_mw",
        "sort": "datetime_beginning_utc",
    })
    out = pd.DataFrame({
        "time_utc": pd.to_datetime(raw["datetime_beginning_utc"], utc=True),
        "area_id": "PJM_" + raw["area"],
        "value_mw": pd.to_numeric(raw[f"{kind}_generation_mw"]),
    })
    # Cross-check: converting PJM's local EPT time ourselves must give the same UTC time.
    local = pd.DataFrame({"t": raw["datetime_beginning_ept"], "area_id": out["area_id"], "row": out.index})
    converted = to_utc(local, "t", "America/New_York").sort_values("row")
    mismatched = int((converted["time_utc"].values != out["time_utc"].values).sum())
    print(f"PJM {kind}: {len(out)} rows, {out['area_id'].nunique()} areas; EPT vs UTC mismatches: {mismatched}")
    # RTO and RFC are totals of the three smallest areas, and OTHER is outside RTO; see the text above.
    out = out[out["area_id"].isin(["PJM_MIDATL", "PJM_WEST", "PJM_SOUTH"])].reset_index(drop=True)
    negative = int((out["value_mw"] < 0).sum())
    out["value_mw"] = out["value_mw"].clip(lower=0)
    print(f"Kept MIDATL, WEST and SOUTH; set {negative} small negative values to 0.")
    return out

print("fetch_pjm_renewables() is ready.")

**(live)** Downloads one year of PJM solar and wind, saves the raw files, prints the areas PJM reports, and runs the checks. It only runs when `RUN_LIVE = True` in the setup box.

In [ ]:
if RUN_LIVE:
    for kind in ["solar", "wind"]:
        pjm = fetch_pjm_renewables(kind, YEAR)
        pjm.to_csv(f"{DATA_DIR}/raw/pjm_{kind}_{YEAR}.csv", index=False)
        print("Areas:", sorted(pjm["area_id"].unique()))
        check_full_year(pjm, YEAR)
        check_values(pjm, kind)
else:
    print("Skipped: RUN_LIVE is False.")

#### MISO (Historical Generation Fuel Mix)

MISO publishes one Excel file per year with hourly generation by fuel for its three regions, **North, Central and South**: [historical_gen_fuel_mix_2023.xlsx](https://docs.misoenergy.org/marketreports/historical_gen_fuel_mix_2023.xlsx). We use the real-time column (`RT Generation State Estimator`).

The time label is `HourEnding` in **Eastern Standard Time** (UTC−5 all year): hour ending 1 is 00:00 to 01:00 EST. This matches Section 5, and it was confirmed on October 25, 2026: the three regions' solar adds up to EIA's MISO solar within 3 MW on average once both are put on the same clock.

In [ ]:
MISO_FUEL_MIX_URL = "https://docs.misoenergy.org/marketreports/historical_gen_fuel_mix_{year}.xlsx"


def fetch_miso_renewables(year):
    # Returns {"solar": df, "wind": df}, each time_utc, area_id (MISO_NORTH/CENTRAL/SOUTH), value_mw.
    path = os.path.join(DATA_DIR, "raw", f"miso_fuel_mix_{year}.xlsx")
    if not os.path.exists(path):
        response = requests.get(MISO_FUEL_MIX_URL.format(year=year), timeout=300)
        response.raise_for_status()
        open(path, "wb").write(response.content)
    raw = pd.read_excel(path, header=4, usecols=range(1, 7))
    raw.columns = ["date", "hour_ending", "region", "fuel", "day_ahead_mw", "real_time_mw"]
    raw = raw.dropna(subset=["date"])
    start = pd.to_datetime(raw["date"]) + pd.to_timedelta(raw["hour_ending"].astype(int) - 1, unit="h")
    raw = raw.assign(local_start=start.dt.strftime("%Y-%m-%d %H:%M"))
    out = {}
    for kind, fuel in [("solar", "Solar"), ("wind", "Wind")]:
        part = raw[raw["fuel"] == fuel]
        df = pd.DataFrame({"local_start": part["local_start"].values,
                           "area_id": "MISO_" + part["region"].str.upper().values,
                           "value_mw": pd.to_numeric(part["real_time_mw"]).values})
        out[kind] = to_utc(df, "local_start", "Etc/GMT+5")
        print(f"MISO {kind}: {len(out[kind])} rows, areas {sorted(out[kind]['area_id'].unique())}")
    return out

print("fetch_miso_renewables() is ready.")

In [9]:
def fetch_renewables():
    raise NotImplementedError(
        "Only PJM is written so far (fetch_pjm_renewables above). ISO-NE, NYISO and MISO come next, "
        "then agree with Yifan on measured area data versus weather-based estimates.")

### Grid test system

Real grid maps, with exact substations and power lines, are kept private for security reasons. Researchers therefore use published "synthetic" grids: made-up buses and lines placed on a real map, with realistic sizes. This step will read the group's chosen test system, keep only the Eastern Interconnection, and use each bus's latitude and longitude to record which area it sits in.

The bus list tells us which area's data belongs to each bus. The line list (from bus, to bus, impedance and flow limit) is what the model uses to build $H$ and $\bar{\mathbf{f}}$.

Yifan wants the **Eastern Interconnection**. Until he confirms a specific case, this code assumes Texas A&M's synthetic **ACTIVSg70k** (about 70,000 buses), stored in MATPOWER format (`ACTIVSg70k.m`). The case file has no bus locations, so they are read from a separate file with `bus_id`, `latitude`, `longitude`. The existing load at each bus becomes its demand weight (Section 9).

**Still to do:** set each bus's `area_id`, i.e. which zone it sits in. That needs zone boundaries for each operator, so it waits until the test system is confirmed.

In [10]:
import re
import numpy as np


def read_matpower_table(text, name):
    # Pull one numeric table, such as mpc.bus or mpc.branch, out of a MATPOWER .m case file.
    match = re.search(rf"mpc\.{name}\s*=\s*\[(.*?)\];", text, re.S)
    if match is None:
        raise ValueError(f"No mpc.{name} table in the case file.")
    rows = []
    for line in match.group(1).splitlines():
        line = line.split("%")[0].strip().rstrip(";")
        if line:
            rows.append([float(v) for v in line.split()])
    return np.array(rows)


def load_grid_test_system(case_path, coords_path):
    # case_path:   MATPOWER case file, e.g. ACTIVSg70k.m (bus and branch tables)
    # coords_path: CSV with bus_id, latitude, longitude (the case file itself has no locations)
    # returns:     buses (bus_id, latitude, longitude, area_id, voltage_kv, load_mw) and
    #              lines (line_id, from_bus, to_bus, reactance_pu, flow_limit_mw), as in Section 4
    text = open(case_path).read()
    bus = read_matpower_table(text, "bus")
    branch = read_matpower_table(text, "branch")
    buses = pd.DataFrame({
        "bus_id": bus[:, 0].astype(int),
        "voltage_kv": bus[:, 9],
        "load_mw": bus[:, 2],      # existing load at the bus, used for the demand weights
    })
    coords = pd.read_csv(coords_path)[["bus_id", "latitude", "longitude"]]
    buses = buses.merge(coords, on="bus_id", how="left")
    missing = int(buses["latitude"].isna().sum())
    if missing:
        print(f"Warning: {missing} buses have no location.")
    buses["area_id"] = None        # filled in by matching each bus to the area it sits in
    in_service = branch[:, 10] == 1
    lines = pd.DataFrame({
        "from_bus": branch[in_service, 0].astype(int),
        "to_bus": branch[in_service, 1].astype(int),
        "reactance_pu": branch[in_service, 3],
        "flow_limit_mw": branch[in_service, 5],   # rateA; 0 means no limit in MATPOWER
    })
    lines.insert(0, "line_id", range(1, len(lines) + 1))
    print(f"Loaded {len(buses)} buses and {len(lines)} lines in service.")
    return buses[["bus_id", "latitude", "longitude", "area_id", "voltage_kv", "load_mw"]], lines


def demand_weights(buses):
    # Demand weight = the bus's existing load. Buses with no load (or negative load) get 0.
    return buses[["bus_id", "area_id"]].assign(weight=buses["load_mw"].clip(lower=0))


print("load_grid_test_system() and demand_weights() are ready (tested on demo data only).")

### Plant locations and sizes (EIA-860)

The capacity weights in Section 9 need every solar and wind plant's location and size. These come from **Form EIA-860**, a free yearly download from [eia.gov/electricity/data/eia860](https://www.eia.gov/electricity/data/eia860/): one ZIP file of Excel sheets per year. Unzip it into `data/raw/eia860/`. Two sheets are used:

- `2___Plant_Y<year>.xlsx`: each plant's code, latitude, longitude and balancing authority.
- `3_1_Generator_Y<year>.xlsx`, sheet `Operable`: each generator's plant code, technology and nameplate capacity in MW.

This function joins them and returns one row per plant and technology, ready for `capacity_weights()`. The plant list is also used below to place each bus in a balancing authority.

In [ ]:
TECHNOLOGIES = {
    "solar": ["Solar Photovoltaic", "Solar Thermal without Energy Storage", "Solar Thermal with Energy Storage"],
    "wind": ["Onshore Wind Turbine", "Offshore Wind Turbine"],
}


def load_eia860(folder, year):
    # Returns (plants, capacity): plants = plant_id, latitude, longitude, ba_code (every plant);
    # capacity = plant_id, latitude, longitude, kind, capacity_mw (solar and wind only).
    plant = pd.read_excel(os.path.join(folder, f"2___Plant_Y{year}.xlsx"), skiprows=1)
    gen = pd.read_excel(os.path.join(folder, f"3_1_Generator_Y{year}.xlsx"), sheet_name="Operable", skiprows=1)
    plants = pd.DataFrame({
        "plant_id": plant["Plant Code"],
        "latitude": pd.to_numeric(plant["Latitude"], errors="coerce"),
        "longitude": pd.to_numeric(plant["Longitude"], errors="coerce"),
        "ba_code": plant["Balancing Authority Code"],
    }).dropna(subset=["latitude", "longitude"])
    parts = []
    for kind, techs in TECHNOLOGIES.items():
        g = gen[gen["Technology"].isin(techs)]
        mw = pd.to_numeric(g["Nameplate Capacity (MW)"], errors="coerce").groupby(g["Plant Code"]).sum()
        parts.append(pd.DataFrame({"plant_id": mw.index, "kind": kind, "capacity_mw": mw.values}))
    capacity = pd.concat(parts).merge(plants[["plant_id", "latitude", "longitude"]], on="plant_id")
    print(f"EIA-860 {year}: {len(plants)} plants; "
          + ", ".join(f"{k} {capacity[capacity.kind == k].capacity_mw.sum():,.0f} MW" for k in TECHNOLOGIES))
    return plants, capacity

print("load_eia860() is ready (not yet tested on the real files).")

### Placing each bus in an area

Each bus needs an `area_id` so it receives the right area's hourly data. For areas that are whole balancing authorities (the EIA-930 data above), there is a simple way to do this without any map files: EIA-860 records the BA of every power plant. Each bus takes the BA that most of its **5 nearest plants** belong to. Plants are everywhere, so this follows BA borders closely, though a bus right on a border can land on the wrong side.

Areas finer than a BA (for example PJM's MIDATL, WEST and SOUTH, or NYISO's zones) need zone boundary maps, which come later.

In [ ]:
def assign_ba_areas(buses, plants, k=5, chunk=500):
    # buses: bus_id, latitude, longitude.  plants: latitude, longitude, ba_code (from load_eia860).
    # Returns buses with area_id = "EIA_" + the most common BA among the k nearest plants.
    p = plants.dropna(subset=["ba_code"]).reset_index(drop=True)
    plat, plon, pba = np.radians(p["latitude"].values), np.radians(p["longitude"].values), p["ba_code"].values
    blat, blon = np.radians(buses["latitude"].values), np.radians(buses["longitude"].values)
    areas = []
    for s in range(0, len(buses), chunk):   # in chunks, so 70,000 buses x 10,000 plants fits in memory
        la, lo = blat[s:s + chunk, None], blon[s:s + chunk, None]
        a = np.sin((plat - la) / 2) ** 2 + np.cos(la) * np.cos(plat) * np.sin((plon - lo) / 2) ** 2
        nearest = np.argsort(a, axis=1)[:, :k]
        areas += [pd.Series(pba[row]).mode().iloc[0] for row in nearest]
    out = buses.copy()
    out["area_id"] = ["EIA_" + a for a in areas]
    print(out["area_id"].value_counts().head(10).to_string())
    return out

print("assign_ba_areas() is ready.")

**(demo)** Four made-up buses and eight made-up plants in two BAs. The two western buses sit among MISO plants and the two eastern ones among PJM plants.

In [ ]:
demo_plants = pd.DataFrame({
    "latitude":  [41.0, 41.2, 40.8, 41.1, 40.9, 40.0, 40.2, 39.9],
    "longitude": [-88.0, -87.8, -88.2, -87.9, -88.1, -80.0, -80.2, -79.9],
    "ba_code":   ["MISO"] * 5 + ["PJM"] * 3,
})
demo_grid_buses = pd.DataFrame({"bus_id": [1, 2, 3, 4], "latitude": [41.0, 40.9, 40.1, 40.0],
                                "longitude": [-88.0, -87.9, -80.1, -80.0]})
display(assign_ba_areas(demo_grid_buses, demo_plants, k=3))

## 9. Splitting area data down to buses

### Why we split

A **bus** is a single point on the grid where power enters or leaves. Zones and regions are areas on a map, and each one contains many buses. The model works bus by bus: it needs the solar, wind and demand at each bus, because that decides how much power flows on each line. Our data, however, only comes as one hourly total per area, such as a zone. So we need a way to turn area totals into per-bus numbers.

### Two ways to get per-bus numbers

Both options use the same **capacity** data from EIA-860 (how big each solar or wind farm is and where it is; this never changes hour to hour). They differ only in where the **hourly pattern** comes from.

| | Where the hourly pattern comes from | Strength | Weakness |
|---|---|---|---|
| **Option A: split a measured area total** (our plan) | The zone's measured hourly output, from the ISO or EIA-930 | Real, measured numbers | Every bus in a zone gets the same pattern |
| **Option B: estimate from weather** (question for Yifan) | Sunshine and wind speed at each bus's exact location | Each bus gets its own local pattern | A modeled estimate, not measured output |

**Option A is the plan.** It is what the data collected so far supports, and it is what the rest of this section builds. **Option B is not being built.** It is an alternative to discuss with Yifan: weather data, for example from NREL (the National Renewable Energy Laboratory), would be converted into estimated output for the capacity near each bus.

The rest of this section describes Option A. The smaller the area we start from, the better it works, because buses in a small area really do see similar weather and similar daily patterns.

### Each data type is split separately

Solar, wind and demand are **not** combined into one weight. Each depends on something different, so each gets its own weight, which is the bus's share of that thing within its area:

| Data type | What the weight measures | Weight for bus $b$ |
|---|---|---|
| Solar | Where the solar farms are | bus's solar capacity ÷ area's solar capacity |
| Wind | Where the wind farms are | bus's wind capacity ÷ area's wind capacity |
| Demand | Where people and businesses use power | bus's load ÷ area's load |

One bus can have a large share of solar but a small share of demand, for example a bus next to a big solar farm in the countryside. A single combined weight would hide that difference. Each type is then split on its own, every hour:

$$\text{solar}_{t,b} = w_b^{\text{solar}} \times \text{Solar}_t^{\text{area}}, \qquad \text{wind}_{t,b} = w_b^{\text{wind}} \times \text{Wind}_t^{\text{area}}, \qquad d_{t,b} = w_b^{\text{demand}} \times D_t^{\text{area}}$$

Within each area and each data type, the weights add up to 1 (100%).

### Where they come together: inside the model

The three values only meet in the model, through each bus's **net injection**, which is the expression inside Constraints 2 and 3:

$$\underbrace{g_{t,b}}_{\text{generators}} + \underbrace{r_{t,b}}_{\text{solar + wind}} - \underbrace{d_{t,b}}_{\text{demand}} - \underbrace{c_b}_{\text{new load}}$$

For example, suppose at noon an area produces 80 MW of solar and 50 MW of wind and uses 200 MW. Bus 1 holds 60% of the area's solar capacity, 20% of its wind capacity and 10% of its load. Bus 1 then gets 48 MW of solar, 10 MW of wind and 20 MW of demand. Before any new load, it has up to 38 MW of extra renewable power to send to the rest of the grid, and the model works out how much new load can be added there before a line reaches its limit.

### Where the capacity numbers come from

**Solar and wind capacity** come from **Form EIA-860**, a yearly survey by the US Energy Information Administration of every power plant in the country with at least 1 MW of capacity. It is a free download from the EIA website, one ZIP file of Excel sheets per year. Two of its sheets matter for us:

- The **plant** sheet gives each plant's ID, latitude, longitude and balancing authority.
- The **generator** sheet gives each generator's plant ID, technology (for example solar photovoltaic or onshore wind) and nameplate capacity in MW.

To turn these into weights, we join the two sheets on the plant ID, keep only solar and wind generators in our region, attach each plant to the **nearest bus**, and add up the capacity at each bus. EIA also publishes a monthly version, EIA-860M, with the same kind of information, if we need something more recent than the yearly release.

One limitation: EIA-860 leaves out installations under 1 MW, so most rooftop solar is missing. That is worth mentioning to Yifan, since rooftop solar is common in some zones.

**Demand weights** come from the base load listed for each bus in the grid test system. If the test system doesn't list one, population near each bus is a common substitute.

### The assumption behind the method

This method assumes every bus in an area follows the **same hourly pattern**, only at a different size: if the area is cloudy, every bus in it gets less solar. That is reasonable for a small zone and much weaker for a huge region, which is why we collect the smallest regions available.

### Turning plant capacity into bus weights

This function attaches each power plant to its nearest bus (by distance on the Earth's surface) and adds up the capacity at each bus. The result is a weight table that the splitting step below can use directly.

In [11]:
import numpy as np

def distance_km(lat1, lon1, lat2, lon2):
    # Distance between points on the Earth's surface (haversine formula).
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def capacity_weights(plants, buses):
    # plants: plant_id, latitude, longitude, capacity_mw   (one technology, e.g. only solar)
    # buses:  bus_id, area_id, latitude, longitude
    # returns: bus_id, area_id, weight   (weight = installed capacity attached to that bus, in MW)
    nearest = []
    for _, plant in plants.iterrows():
        d = distance_km(plant["latitude"], plant["longitude"], buses["latitude"], buses["longitude"])
        nearest.append(buses["bus_id"].iloc[int(np.argmin(d))])
    attached = plants.assign(bus_id=nearest)
    per_bus = attached.groupby("bus_id")["capacity_mw"].sum()
    out = buses[["bus_id", "area_id"]].copy()
    out["weight"] = out["bus_id"].map(per_bus).fillna(0.0)
    return out

print("capacity_weights() is ready.")

capacity_weights() is ready.


**(demo)** Three made-up buses in one area and four made-up solar plants. Bus 1 ends up with 60 MW, bus 2 with 30 MW and bus 3 with 10 MW, so they get 60%, 30% and 10% of the area's solar.

In [12]:
demo_bus_list = pd.DataFrame({
    "bus_id": [1, 2, 3],
    "area_id": ["A", "A", "A"],
    "latitude": [40.40, 40.90, 41.50],
    "longitude": [-86.90, -86.20, -85.50],
})
demo_solar_plants = pd.DataFrame({
    "plant_id": [101, 102, 103, 104],
    "latitude": [40.42, 40.38, 40.88, 41.52],
    "longitude": [-86.88, -86.95, -86.25, -85.48],
    "capacity_mw": [40, 20, 30, 10],
})

solar_weights = capacity_weights(demo_solar_plants, demo_bus_list)
solar_weights["share_%"] = 100 * solar_weights["weight"] / solar_weights["weight"].sum()
display(solar_weights)

,bus_id,area_id,weight,share_%
0,1,A,60,60.0
1,2,A,30,30.0
2,3,A,10,10.0


### Splitting each hour's total

This function takes an area's hourly totals and the weights, scales the weights so each area's shares add up to 1, and multiplies. A second function adds the bus values back up and confirms they equal the original totals.

In [13]:
def split_to_buses(area_totals, weights):
    # area_totals: time_utc, area_id, value_mw
    # weights:     bus_id, area_id, weight
    w = weights.copy()
    w["share"] = w["weight"] / w.groupby("area_id")["weight"].transform("sum")
    merged = area_totals.merge(w[["bus_id", "area_id", "share"]], on="area_id")
    merged["value_mw"] = merged["value_mw"] * merged["share"]
    return merged[["time_utc", "bus_id", "area_id", "value_mw"]]


def check_totals_match(area_totals, bus_values):
    # Add bus values back up per area and hour; they must equal the original totals.
    back = bus_values.groupby(["time_utc", "area_id"])["value_mw"].sum().reset_index()
    both = area_totals.merge(back, on=["time_utc", "area_id"], suffixes=("_area", "_buses"))
    biggest_gap = (both["value_mw_area"] - both["value_mw_buses"]).abs().max()
    if biggest_gap < 1e-6:
        print(f"Totals match for all {len(both)} area-hours.")
    else:
        print(f"Totals do NOT match: biggest gap is {biggest_gap:.4f} MW.")

print("split_to_buses() and check_totals_match() are ready.")

split_to_buses() and check_totals_match() are ready.


**(demo)** Area A has three buses with weights 5, 3 and 2, so they get 50%, 30% and 20% of its total. Area B has one bus, which gets everything.

In [14]:
demo_hours = pd.to_datetime(["2023-01-01 00:00", "2023-01-01 01:00"], utc=True)
demo_totals = pd.DataFrame({
    "time_utc": list(demo_hours) * 2,
    "area_id": ["A", "A", "B", "B"],
    "value_mw": [1000, 900, 400, 500],
})
demo_weights = pd.DataFrame({
    "bus_id": [1, 2, 3, 4],
    "area_id": ["A", "A", "A", "B"],
    "weight": [5, 3, 2, 1],
})

demo_buses = split_to_buses(demo_totals, demo_weights)
display(demo_buses)
check_totals_match(demo_totals, demo_buses)

,time_utc,bus_id,area_id,value_mw
0,2023-01-01 00:00:00+00:00,1,A,500.0
1,2023-01-01 00:00:00+00:00,2,A,300.0
2,2023-01-01 00:00:00+00:00,3,A,200.0
3,2023-01-01 01:00:00+00:00,1,A,450.0
4,2023-01-01 01:00:00+00:00,2,A,270.0
5,2023-01-01 01:00:00+00:00,3,A,180.0
6,2023-01-01 00:00:00+00:00,4,B,400.0
7,2023-01-01 01:00:00+00:00,4,B,500.0


Totals match for all 4 area-hours.


**(demo)** The worked example from above: solar, wind and demand for one area at noon, each split with its own weights, then combined into each bus's net power before any new load. Bus 1 gets 48 MW of solar, 10 MW of wind and 20 MW of demand, leaving 38 MW to spare.

In [15]:
noon = pd.to_datetime(["2023-07-01 16:00"], utc=True)   # noon in Indiana in July (daylight time, UTC-4) is 16:00 UTC

area_noon = {"solar": 80, "wind": 50, "demand": 200}      # area totals in MW
bus_weights = {                                             # each type has its own weights
    "solar":  [60, 30, 10],   # MW of solar capacity near buses 1, 2, 3
    "wind":   [20, 80, 0],    # wind capacity shares (%)
    "demand": [10, 30, 60],   # load shares (%)
}

per_bus = pd.DataFrame({"bus_id": [1, 2, 3]})
for kind, total in area_noon.items():
    totals = pd.DataFrame({"time_utc": noon, "area_id": ["A"], "value_mw": [total]})
    weights = pd.DataFrame({"bus_id": [1, 2, 3], "area_id": "A", "weight": bus_weights[kind]})
    per_bus[f"{kind}_mw"] = split_to_buses(totals, weights)["value_mw"].values

per_bus["net_mw"] = per_bus["solar_mw"] + per_bus["wind_mw"] - per_bus["demand_mw"]
display(per_bus)
print("Positive net = the bus can send extra renewable power to the grid; negative = it needs power from elsewhere.")

,bus_id,solar_mw,wind_mw,demand_mw,net_mw
0,1,48.0,10.0,20.0,38.0
1,2,24.0,40.0,60.0,4.0
2,3,8.0,0.0,120.0,-112.0


Positive net = the bus can send extra renewable power to the grid; negative = it needs power from elsewhere.


## 10. Building the shared files

This step combines every source into the files described in Section 4: `areas.csv` plus one file per data type (`demand.csv`, `solar.csv`, `wind.csv`) in `data/clean/`, all in the same three columns.

Where a finer source exists it **replaces** the coarser one, so nothing is counted twice:

| Data | Finer source used | Replaces |
|---|---|---|
| MISO demand | 6 EIA-930 subregions (they add up to the MISO total within 70 MW) | `EIA_MISO` |
| PJM solar and wind | PJM's MIDATL, WEST and SOUTH zones | `EIA_PJM` |
| MISO solar and wind | MISO's NORTH, CENTRAL and SOUTH regions | `EIA_MISO` |

Every other area is one whole balancing authority from EIA-930. It reads the files already downloaded and cleaned above, so it needs no internet.

In [ ]:
ISO_NAMES = {"MISO": "MISO", "PJM": "PJM", "ISNE": "ISO-NE", "NYIS": "NYISO", "SWPP": "SPP"}


def describe_area(area_id):
    # One row of areas.csv, worked out from the area_id prefix.
    prefix, name = area_id.split("_", 1)
    if prefix == "EIA":
        return {"operator": ISO_NAMES.get(name, "none (utility BA)"), "ba": name, "zone": "",
                "resolution": "balancing authority", "source": "EIA-930"}
    if prefix == "MISO" and name.isdigit():
        return {"operator": "MISO", "ba": "MISO", "zone": name, "resolution": "subregion", "source": "EIA-930 subregion"}
    if prefix == "MISO":
        return {"operator": "MISO", "ba": "MISO", "zone": name, "resolution": "region", "source": "MISO fuel mix"}
    if prefix == "PJM":
        return {"operator": "PJM", "ba": "PJM", "zone": name, "resolution": "zone", "source": "PJM Data Miner 2"}
    raise ValueError(f"Unknown area prefix: {area_id}")


def build_shared_files(year):
    clean = os.path.join(DATA_DIR, "clean")
    read = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), parse_dates=["time_utc"])
    eia = {k: read(f"clean/eia930_ba_{k}_{year}.csv") for k in ["demand", "solar", "wind"]}

    miso = clean_hourly(read(f"raw/eia930_miso_subregion_demand_{year}.csv"), "demand", year)
    pjm = {}
    for k in ["solar", "wind"]:
        raw = read(f"raw/pjm_{k}_{year}.csv")
        raw = raw[raw["area_id"].isin(["PJM_MIDATL", "PJM_WEST", "PJM_SOUTH"])]
        pjm[k] = clean_hourly(raw, k, year)

    miso_rw = {k: clean_hourly(v, k, year) for k, v in fetch_miso_renewables(year).items()}

    data = {
        "demand": pd.concat([eia["demand"][eia["demand"]["area_id"] != "EIA_MISO"], miso]),
        "solar": pd.concat([eia["solar"][~eia["solar"]["area_id"].isin(["EIA_PJM", "EIA_MISO"])], pjm["solar"], miso_rw["solar"]]),
        "wind": pd.concat([eia["wind"][~eia["wind"]["area_id"].isin(["EIA_PJM", "EIA_MISO"])], pjm["wind"], miso_rw["wind"]]),
    }
    for kind, df in data.items():
        df = df.sort_values(["area_id", "time_utc"])
        df.to_csv(os.path.join(clean, f"{kind}.csv"), index=False)
        print(f"{kind}.csv: {df['area_id'].nunique()} areas, {len(df):,} rows")
        check_full_year(df, year)
        check_values(df, kind)

    ids = sorted(set().union(*[set(df["area_id"]) for df in data.values()]))
    areas = pd.DataFrame([{"area_id": a, "interconnection": "Eastern", **describe_area(a),
                           **{f"has_{k}": a in set(data[k]["area_id"]) for k in data}} for a in ids])
    areas.to_csv(os.path.join(clean, "areas.csv"), index=False)
    print(f"areas.csv: {len(areas)} areas")
    return areas

print("build_shared_files() is ready.")

In [ ]:
if os.path.exists(os.path.join(DATA_DIR, "clean", f"eia930_ba_demand_{YEAR}.csv")):
    areas = build_shared_files(YEAR)
    display(areas)
else:
    print("Skipped: run the live download boxes above first.")

## 11. Saving the model files

The model numbers hours from 1 to 8,760 instead of using dates. This function swaps each UTC timestamp for its hour number and saves the result.

In [16]:
def save_for_model(bus_values, year, filename, value_name):
    start = pd.Timestamp(f"{year}-01-01 00:00", tz="UTC")
    out = bus_values.copy()
    out["hour"] = ((out["time_utc"] - start) / pd.Timedelta(hours=1)).astype(int) + 1
    out = out.rename(columns={"value_mw": value_name})[["hour", "bus_id", value_name]]
    out.to_csv(os.path.join(DATA_DIR, "model", filename), index=False)
    return out

**(demo)** Saves the demo result as a sample of what `bus_demand.csv` will look like.

In [17]:
sample = save_for_model(demo_buses, 2023, "sample_bus_demand.csv", "demand_mw")
display(sample)
print("Saved to", os.path.join(DATA_DIR, "model", "sample_bus_demand.csv"))

,hour,bus_id,demand_mw
0,1,1,500.0
1,1,2,300.0
2,1,3,200.0
3,2,1,450.0
4,2,2,270.0
5,2,3,180.0
6,1,4,400.0
7,2,4,500.0


Saved to data/model/sample_bus_demand.csv


## 12. Open questions for Yifan

1. Which grid test system (bus and line data) does the group use, and where are its files?
2. Which year should the data cover?
3. ~~Option A or Option B?~~ **Answered (October 23, 2026):** start with Option A (split measured zone totals by installed capacity) and add on later.
4. Does the file layout in Section 4 work for the model?
5. EIA-860 only lists plants of 1 MW or more, so most rooftop solar is missing from the solar weights. Is that acceptable, or should we account for it?